# 06 · Previsão

Usamos o modelo escolhido para estimar o preço pedido de um apartamento em Lisboa (até 2,5 milhões de €).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

pd.set_option("display.max_colwidth", None)   # mostrar os textos inteiros nas tabelas

pacote = joblib.load(Path("..") / "models" / "modelos.joblib")
modelo = pacote["modelos"][pacote["escolhido"]]
margem_baixa, margem_alta = pacote["margem"]

# Dados usados no treino: para saber o centro de cada freguesia e as áreas que o modelo conhece
dados = pd.read_csv(Path("..") / "data" / "processed" / "dados_modelo.csv")
centro_freguesia = dados.groupby("freguesia")[["latitude", "longitude"]].median()
area_min, area_max = dados["area_m2"].min(), dados["area_m2"].max()
print(f"Modelo: {pacote['escolhido']}")

Modelo: XGBoost


## A função de estimativa

Recebe a descrição da casa e faz as mesmas contas do notebook 03 (log da área e distância ao Chiado). Se não souberes as coordenadas, escreve `None`: usa o centro da freguesia.

Além da estimativa e do intervalo, mostra o preço por m² da estimativa, para se ter uma ideia do valor.

In [2]:
def estimar_preco(casa):
    casa = dict(casa)   # cópia, para não alterar o original
    if casa["latitude"] is None:
        casa["latitude"], casa["longitude"] = centro_freguesia.loc[casa["freguesia"]]

    casa["log_area"] = np.log(casa["area_m2"])
    dy = (casa["latitude"] - 38.7107) * 110.57    # as mesmas contas do notebook 03
    dx = (casa["longitude"] - (-9.1433)) * 86.9
    casa["distancia_centro_km"] = np.sqrt(dx**2 + dy**2)

    tabela = pd.DataFrame([casa])[modelo.feature_names_in_]   # as colunas pela ordem do treino
    valor = modelo.predict(tabela)[0]

    resultado = {
        "Estimativa": f"{valor / 1000:.0f} mil €",
        "Intervalo provável (80%)": f"{valor * margem_baixa / 1000:.0f} a {valor * margem_alta / 1000:.0f} mil €",
        "Preço por m²": f"{valor / casa['area_m2']:,.0f} €".replace(",", " "),
    }
    if not area_min <= casa["area_m2"] <= area_max:
        resultado["Aviso"] = f"Nos dados, as áreas vão de {area_min} a {area_max} m²: esta estimativa é pouco fiável."
    return pd.Series(resultado).to_frame("Resultado")

## Exemplo: T2 de 90 m² em Arroios

| Campo | Valores possíveis |
|---|---|
| `estado` | `"good"` (bom estado), `"renew"` (para recuperar), `"newdevelopment"` (novo) |
| `tipo` | `"flat"` (apartamento), `"duplex"`, `"penthouse"` (cobertura), `"studio"` (estúdio) |
| extras | `1` = sim, `0` = não |

In [3]:
casa = {
    "freguesia": "Arroios", "latitude": 38.7280, "longitude": -9.1350,
    "area_m2": 90, "quartos": 2, "casas_banho": 2, "andar": 3,
    "estado": "good", "tipo": "flat",
    "elevador": 1, "empreendimento_novo": 0, "terraco": 0, "piscina": 0, "jardim": 0,
    "ar_condicionado": 1, "arrecadacao": 0, "estacionamento": 0,
}
estimar_preco(casa)

,Resultado
Estimativa,580 mil €
Intervalo provável (80%),462 a 727 mil €
Preço por m²,6 445 €


## A tua vez

Muda os valores e corre a célula. Antes, corre as células de cima (no VS Code: "Run All").

In [4]:
minha_casa = {
    "freguesia": "Avenidas Novas", "latitude": None, "longitude": None,
    "area_m2": 80, "quartos": 2, "casas_banho": 2, "andar": 4,
    "estado": "good", "tipo": "flat",
    "elevador": 0, "empreendimento_novo": 0, "terraco": 0, "piscina": 0, "jardim": 0,
    "ar_condicionado": 0, "arrecadacao": 1, "estacionamento": 0,
}
estimar_preco(minha_casa)

,Resultado
Estimativa,577 mil €
Intervalo provável (80%),460 a 723 mil €
Preço por m²,7 218 €


Lembra-te: em 1 de cada 5 casas o preço real fica fora do intervalo, e casas invulgarmente baratas para as suas características tendem a sair estimadas a mais.